# 电商用户复购与流失分析（UCI Online Retail II）
数据：https://archive.ics.uci.edu/dataset/502/online+retail+ii （CC BY 4.0，引用 Chen, D. 2012）

流程：读取体检 → 清洗 → EDA → RFM 分层 → 留存 → 90 天复购预测 → 业务建议
使用方法：把 online_retail_II.xlsx 放在本文件同一目录，从上到下依次运行。
带 【TODO】 的地方是需要你自己看结果、写结论的地方。

In [ ]:
# 0. 配置与导入
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "online_retail_II.xlsx"    # 也支持 .csv
CUTOFF = pd.Timestamp("2011-09-09")    # 预测任务的切分日：只用这天之前的数据做特征
HORIZON = 90                           # 预测未来 90 天内是否再次购买
os.makedirs("figures", exist_ok=True)
plt.rcParams["figure.dpi"] = 110

In [ ]:
# 1. 读取与体检
if DATA_PATH.endswith(".csv"):
    raw = pd.read_csv(DATA_PATH)
else:
    sheets = pd.read_excel(DATA_PATH, sheet_name=None)   # 文件含多个工作表，合并
    raw = pd.concat(sheets.values(), ignore_index=True)

raw.columns = [c.strip() for c in raw.columns]
raw = raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice",
                          "Customer ID": "CustomerID"})
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
raw["InvoiceNo"] = raw["InvoiceNo"].astype(str)

print("原始行数:", len(raw))
print("时间范围:", raw["InvoiceDate"].min(), "→", raw["InvoiceDate"].max())
print("\n各列缺失率:\n", raw.isna().mean().round(3))
print("\n完全重复行:", raw.duplicated().sum())
print("退货行(发票号以C开头):", raw["InvoiceNo"].str.startswith("C").sum())
# 【TODO】把缺失率、重复行数、退货行数记下来，写进 README 的"数据质量"一节

In [ ]:
# 2. 清洗（每一步都记录行数，简历里的"清洗约X条"来自这里）
log = {"原始行数": len(raw)}

df = raw.drop_duplicates()
log["去重后"] = len(df)

df = df.dropna(subset=["CustomerID"]).copy()
df["CustomerID"] = df["CustomerID"].astype(int)
log["去掉缺失客户ID后"] = len(df)

# 退货单独保存，用于后面算退货率
df["is_cancel"] = df["InvoiceNo"].str.startswith("C")
returns = df[df["is_cancel"]].copy()

# 正常销售：数量>0、单价>0；并剔除 POST/BANK CHARGES 等非商品编码（商品编码以5位数字开头）
sales = df[(~df["is_cancel"]) & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
sales = sales[sales["StockCode"].astype(str).str.match(r"^\d{5}")]
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
log["有效销售行"] = len(sales)

print(pd.Series(log))
print("有效客户数:", sales["CustomerID"].nunique())
print("总销售额(£):", round(sales["Revenue"].sum()))

In [ ]:
# 3. EDA：月度趋势与客户集中度
sales["Month"] = sales["InvoiceDate"].dt.to_period("M")
monthly = sales.groupby("Month").agg(
    revenue=("Revenue", "sum"),
    orders=("InvoiceNo", "nunique"),
    customers=("CustomerID", "nunique"),
)
ax = monthly.plot(subplots=True, figsize=(9, 7), title="月度销售额 / 订单数 / 活跃客户数", legend=False)
plt.tight_layout()
plt.savefig("figures/01_monthly_trend.png")
plt.show()

cust_rev = sales.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
top10 = cust_rev.head(int(len(cust_rev) * 0.1)).sum() / cust_rev.sum()
print(f"前 10% 客户贡献了 {top10:.1%} 的销售额")
print("国家分布(按销售额占比):\n", (sales.groupby("Country")["Revenue"].sum()
      .sort_values(ascending=False).head(5) / sales["Revenue"].sum()).round(3))
# 【TODO】观察哪几个月销售额最高、是否有季节性（礼品类商品常在 Q4 冲高）

In [ ]:
# 4. RFM 分层（描述性分析，用全量数据）
snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = sales.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
)

# 用 rank 再分箱，避免 Frequency 大量相同值导致分箱失败
rfm["R"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)


def segment(row):
    r, f = row["R"], row["F"]
    if r >= 4 and f >= 4:
        return "高价值核心"
    if r >= 3 and f >= 3:
        return "稳定活跃"
    if r >= 4 and f <= 2:
        return "新客/潜力"
    if r <= 2 and f >= 4:
        return "流失风险(曾高频)"
    if r <= 2:
        return "沉睡"
    return "一般"


rfm["Segment"] = rfm.apply(segment, axis=1)

seg = rfm.groupby("Segment").agg(
    customers=("Monetary", "size"),
    revenue=("Monetary", "sum"),
    avg_recency=("Recency", "mean"),
    avg_freq=("Frequency", "mean"),
)
seg["客户占比"] = (seg["customers"] / seg["customers"].sum()).round(3)
seg["收入占比"] = (seg["revenue"] / seg["revenue"].sum()).round(3)
seg = seg.sort_values("收入占比", ascending=False)
print(seg)

seg[["客户占比", "收入占比"]].plot.bar(figsize=(8, 4), title="各客户分层：客户占比 vs 收入占比")
plt.tight_layout()
plt.savefig("figures/02_rfm_segments.png")
plt.show()
# 【TODO】写出一句结论：占 X% 的客户贡献了 Y% 的收入

In [ ]:
# 5. 留存分析（按首购月份的 cohort）
first_month = sales.groupby("CustomerID")["InvoiceDate"].min().dt.to_period("M").rename("Cohort")
sales = sales.join(first_month, on="CustomerID")
sales["MonthIdx"] = ((sales["Month"].dt.year - sales["Cohort"].dt.year) * 12
                     + (sales["Month"].dt.month - sales["Cohort"].dt.month))

cohort = sales.groupby(["Cohort", "MonthIdx"])["CustomerID"].nunique().unstack(fill_value=0)

# 数据在月中截止，最后一个月不完整：只用到最后一个"完整月份"
last_full = sales["Month"].max() - 1
cohort = cohort[cohort.index <= last_full]
retention = cohort.div(cohort[0], axis=0)

# 关键：还没到观察时间的格子必须是空值(NaN)，不能当成"留存为0"，否则平均留存会被严重拉低
max_idx = (last_full.year - cohort.index.year) * 12 + (last_full.month - cohort.index.month)
for k in retention.columns:
    retention.loc[np.asarray(max_idx < k), k] = np.nan

show = retention.iloc[:, :13]   # 首购后 0~12 个月
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(show.values, cmap="Blues", vmin=0, vmax=0.6, aspect="auto")
ax.set_xticks(range(show.shape[1]))
ax.set_yticks(range(show.shape[0]))
ax.set_yticklabels([str(i) for i in show.index], fontsize=7)
ax.set_xlabel("首购后第 N 个月")
ax.set_title("月度留存率热力图（按首购月份）")
fig.colorbar(im)
plt.tight_layout()
plt.savefig("figures/03_retention_heatmap.png")
plt.show()

print("各 cohort 平均留存率(第1/3/6/12个月，只统计已能观察到的 cohort):")
print(retention.iloc[:, [1, 3, 6, 12]].mean().round(3))
print("\n去掉 2009-12 首个 cohort（含数据开始前就已是老客户的人）后:")
print(retention.iloc[1:, [1, 3, 6, 12]].mean().round(3))
# 注意：第一个 cohort（2009-12）包含数据开始前就已是老客户的人，留存会偏高，所以上面同时给了"去掉它"的结果
# 【TODO】找出留存下降最陡的月份，思考原因

In [ ]:
# 6. 复购预测：构造特征（只用 CUTOFF 之前的数据，防止数据泄漏）
end_date = sales["InvoiceDate"].max()
assert CUTOFF + pd.Timedelta(days=HORIZON) <= end_date + pd.Timedelta(days=1), \
    "切分日 + 90 天超出数据范围，请把 CUTOFF 提前"

hist = sales[sales["InvoiceDate"] < CUTOFF]
future = sales[(sales["InvoiceDate"] >= CUTOFF)
               & (sales["InvoiceDate"] < CUTOFF + pd.Timedelta(days=HORIZON))]

g = hist.groupby("CustomerID")
feat = g.agg(
    recency=("InvoiceDate", lambda x: (CUTOFF - x.max()).days),
    frequency=("InvoiceNo", "nunique"),
    monetary=("Revenue", "sum"),
    n_products=("StockCode", "nunique"),
    first_date=("InvoiceDate", "min"),
)
feat["tenure"] = (CUTOFF - feat["first_date"]).dt.days
feat["avg_order_value"] = feat["monetary"] / feat["frequency"]

# 平均购买间隔（只有一次购买的客户没有间隔：用中位数填充，并加标记）
orders = (hist.groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"].min()
          .reset_index().sort_values(["CustomerID", "InvoiceDate"]))
orders["gap"] = orders.groupby("CustomerID")["InvoiceDate"].diff().dt.days
feat["avg_gap"] = orders.groupby("CustomerID")["gap"].mean()
feat["is_single_order"] = (feat["frequency"] == 1).astype(int)
feat["avg_gap"] = feat["avg_gap"].fillna(feat["avg_gap"].median())

# 退货次数（同样只用切分日之前）
ret_hist = returns[returns["InvoiceDate"] < CUTOFF] if "InvoiceDate" in returns else returns.iloc[0:0]
feat["return_cnt"] = ret_hist.groupby("CustomerID")["InvoiceNo"].nunique()
feat["return_cnt"] = feat["return_cnt"].fillna(0)

# 标签：切分日后 90 天内是否再次购买
feat["y"] = feat.index.isin(future["CustomerID"].unique()).astype(int)
feat = feat.drop(columns=["first_date"])

# 偏态特征取对数（对逻辑回归有帮助）
for c in ["monetary", "avg_order_value", "frequency", "n_products"]:
    feat[c] = np.log1p(feat[c])

print("样本客户数:", len(feat))
print("90 天内复购比例（正样本率）:", round(feat["y"].mean(), 3))
# 【TODO】记录正样本率；它同时是"随机猜"的基线，PR-AUC 要和它比

In [ ]:
# 7. 建模与评估：逻辑回归 vs 随机森林
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

features = [c for c in feat.columns if c != "y"]
X, y = feat[features], feat["y"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

models = {
    "LogisticRegression": make_pipeline(StandardScaler(),
                                        LogisticRegression(max_iter=2000, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                           class_weight="balanced", random_state=42, n_jobs=-1),
}

results, probs = [], {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    probs[name] = p
    results.append({"模型": name,
                    "AUC": round(roc_auc_score(y_te, p), 3),
                    "PR-AUC": round(average_precision_score(y_te, p), 3)})
print(pd.DataFrame(results))
print("基线(随机猜) PR-AUC ≈ 正样本率 =", round(y_te.mean(), 3))

# 特征重要性
rf = models["RandomForest"]
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
imp.plot.barh(figsize=(7, 4), title="随机森林特征重要性")
plt.tight_layout()
plt.savefig("figures/04_feature_importance.png")
plt.show()
# 【TODO】哪个特征最重要？是否符合业务直觉（比如 recency、frequency）？

In [ ]:
# 8. 业务建议：只触达概率最高的客户，能覆盖多少复购者？
best = max(results, key=lambda r: r["PR-AUC"])["模型"]
res = pd.DataFrame({"p": probs[best], "y": y_te.values}).sort_values("p", ascending=False)
total_pos = res["y"].sum()

rows = []
for pct in [0.1, 0.2, 0.3, 0.5]:
    k = int(len(res) * pct)
    captured = res["y"].iloc[:k].sum() / total_pos
    rows.append({"触达最高概率的前": f"{pct:.0%}", "覆盖的复购客户": f"{captured:.1%}"})
print(f"使用模型: {best}")
print(pd.DataFrame(rows))
# 【TODO】把这张表写成一句建议：
# "只对预测概率最高的 X% 客户发放优惠，即可覆盖约 Y% 的将复购客户，
#  相比全量发放可节省约 (1-X) 的营销触达"（注意：这是对公开数据的分析推断，不是实际业务结果）

In [ ]:
# 9. 简历用数字汇总（把这里打印出的真实数字，填回简历的【】里）
print("=" * 40)
print("清洗前后行数:", log)
print("有效客户数:", sales["CustomerID"].nunique())
print("前10%客户收入占比:", f"{top10:.1%}")
print("RFM 分层:\n", seg[["客户占比", "收入占比"]])
print("模型结果:\n", pd.DataFrame(results))
print("Top 概率覆盖表:\n", pd.DataFrame(rows))